In [1]:
# !modal volume create prj-vol
# !modal volume create data-vol
# !modal volume create models-vol

# !modal secret create general WANDB_API_KEY=13ad03bbf5464ebb2b5f4532464556bcc17811a2 HUGGINGFACEHUB_API_TOKEN=REMOVEDRpSsYkbcJiDfMtdEIyrVZPSQlKWKJooDBL

In [2]:
import os
os.environ['HF_HOME'] = "/mnt/models-vol/cache_huggingface"

In [3]:
def setup_environment() -> None:
    import os
    from dotenv import load_dotenv
    _ = load_dotenv()

    from huggingface_hub import login
    login(token=os.environ['HUGGINGFACEHUB_API_TOKEN'])
   
setup_environment()

In [4]:
cd /mnt/prj-vol

/__modal/volumes/vo-KmPbOHyRAQmNKh7dlOw3Is


In [5]:
# !git clone https://github.com/neuphonic/neutts.git

In [5]:
cd /mnt/prj-vol/neutts

/__modal/volumes/vo-KmPbOHyRAQmNKh7dlOw3Is/neutts


In [16]:
%%writefile /mnt/prj-vol/neutts/examples/finetune.py
import warnings

import re
import os
import torch
import phonemizer

from fire import Fire
from omegaconf import OmegaConf
from functools import partial
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from loguru import logger as LOGGER
from datasets import load_dataset

def setup_environment() -> None:
    from dotenv import load_dotenv
    _ = load_dotenv()

    from huggingface_hub import login
    login(token=os.environ['HUGGINGFACEHUB_API_TOKEN'])

setup_environment()

warnings.filterwarnings("ignore")

os.environ['TOKENIZERS_PARALLELISM'] = 'False'


ACRONYM = re.compile(r"(?:[a-zA-Z]\.){2,}")
ACRONYM_NO_PERIOD = re.compile(r"(?:[A-Z]){2,}")


def data_filter(sample):
    text = sample["text"]

    if len(text) == 0:
        return False

    if re.search(r"\d", text):
        return False

    if re.search(ACRONYM, text) or re.search(ACRONYM_NO_PERIOD, text):
        return False

    if text[-1] not in ".,?!":
        return False

    if "£" in text or "$" in text:
        return False

    return True


def preprocess_sample(sample, tokenizer, max_len, g2p):

    # get special tokens
    speech_gen_start = tokenizer.convert_tokens_to_ids("<|SPEECH_GENERATION_START|>")
    ignore_index = -100  # this is from LLaMA

    # unpack sample
    vq_codes = sample["codes"]
    text = sample["text"]

    # phonemize
    phones = g2p.phonemize([text])

    # SAFE CHECK
    if not phones or not phones[0]:
        LOGGER.warning(f"⚠️ Empty phonemization output for sample: {sample['__key__']} text={text}")
        return None

    phones = phones[0].split()
    phones = " ".join(phones)

    codes_str = "".join([f"<|speech_{i}|>" for i in vq_codes])

    # get chat format
    chat = f"""user: Convert the text to speech:<|TEXT_PROMPT_START|>{phones}<|TEXT_PROMPT_END|>\nassistant:<|SPEECH_GENERATION_START|>{codes_str}<|SPEECH_GENERATION_END|>"""  # noqa
    ids = tokenizer.encode(chat)

    # pad to make seq len
    if len(ids) < max_len:
        ids = ids + [tokenizer.pad_token_id] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    # convert to tensor
    input_ids = torch.tensor(ids, dtype=torch.long)

    labels = torch.full_like(input_ids, ignore_index)
    speech_gen_start_idx = (input_ids == speech_gen_start).nonzero(as_tuple=True)[0]
    if len(speech_gen_start_idx) > 0:
        speech_gen_start_idx = speech_gen_start_idx[0]
        labels[speech_gen_start_idx:] = input_ids[speech_gen_start_idx:]

    # create attention mask
    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    # return in hf format
    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
    }


def main(config_fpath: str):

    # load config
    print(f"Loading config from {config_fpath}")
    config = OmegaConf.load(config_fpath)
    checkpoints_dir = os.path.join(config.save_root, config.run_name)
    LOGGER.info(f"Logging to: {checkpoints_dir}")

    restore_from = config.restore_from

    print(f"Loading checkpoint from {restore_from}")
    tokenizer = AutoTokenizer.from_pretrained(restore_from)
    model = AutoModelForCausalLM.from_pretrained(restore_from, torch_dtype="auto")

    g2p = phonemizer.backend.EspeakBackend(
        language="vi",
        preserve_punctuation=True,
        with_stress=True,
        words_mismatch="ignore",
        language_switch="remove-flags",
    )
    partial_preprocess = partial(
        preprocess_sample,
        tokenizer=tokenizer,
        max_len=config.max_seq_len,
        g2p=g2p,
    )

    from datasets import load_from_disk

    prepared_data_dir = '/mnt/data-vol/phoaudiobook_codec_train_prepared'
    
    if not os.path.exists(prepared_data_dir):

        phoaudiobook_codec_dataset = load_from_disk(
            "/mnt/data-vol/phoaudiobook_codec",
            # split="train[:2000]",
        )
        phoaudiobook_codec_dataset_train = phoaudiobook_codec_dataset['train']
    
        phoaudiobook_codec_dataset_train = phoaudiobook_codec_dataset_train.remove_columns(["audio"])
    
        # emilia_dataset = emilia_dataset.filter(data_filter).map(
        #     partial_preprocess, remove_columns=["text", "codes"]
        # )
        
        phoaudiobook_codec_dataset_train = phoaudiobook_codec_dataset_train.map(
            partial_preprocess, 
            remove_columns=["text", "codes"]
        )
    
        phoaudiobook_codec_dataset_train.save_to_disk("/mnt/data-vol/phoaudiobook_codec_train_prepared")

    
    phoaudiobook_codec_dataset_train_prepared = load_from_disk(prepared_data_dir)

    training_args = TrainingArguments(
        output_dir=checkpoints_dir,
        do_train=True,
        learning_rate=config.lr,
        # max_steps=config.max_steps,
        num_train_epochs=config.num_train_epochs,
        bf16=True,
        per_device_train_batch_size=config.per_device_train_batch_size,
        auto_find_batch_size=config.auto_find_batch_size,
        warmup_ratio=config.warmup_ratio,
        save_steps=config.save_steps,
        logging_steps=config.logging_steps,
        save_strategy="steps",
        ignore_data_skip=True,
        dataloader_drop_last=True,
        remove_unused_columns=False,
        torch_compile=True,
        dataloader_num_workers=64,
        report_to="wandb"

    )

    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=phoaudiobook_codec_dataset_train_prepared,
        data_collator=default_data_collator,
    )
    trainer.train()
    trainer.save_model(checkpoints_dir)


if __name__ == "__main__":
    Fire(main)

Overwriting /mnt/prj-vol/neutts/examples/finetune.py


In [7]:
%%writefile /mnt/prj-vol/neutts/examples/finetune_config.yaml
# run info
restore_from: "neuphonic/neutts-air"
save_root: "/mnt/prj-vol/neutts/exps/20260226"
run_name: "neutts-finetune-phoaudiobook"

# model info
codebook_size: 65536 # xcodec
max_seq_len: 2048
lr: 0.00004
lr_scheduler_type: "cosine"
warmup_ratio: 0.00

# train info
per_device_train_batch_size: 64
auto_find_batch_size: true
max_steps: 10000
num_train_epochs: 10
logging_steps: 100
# save_steps: 20000
save_strategy: steps
save_steps: 1000
seed: 202602

Overwriting /mnt/prj-vol/neutts/examples/finetune_config.yaml


In [8]:
%%writefile /mnt/prj-vol/neutts/download_data.py
from datasets import load_dataset

phoaudiobook_codec_ds = load_dataset("namfam/phoaudiobook_codec")
ds_local_path = "/mnt/data-vol/phoaudiobook_codec"

import os
if not os.path.exists(ds_local_path):
    phoaudiobook_codec_ds.save_to_disk(ds_local_path)

Overwriting /mnt/prj-vol/neutts/download_data.py


In [8]:
!apt install espeak-ng -y




The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 50 not upgraded.
Need to get 4829 kB of archives.
After this operation, 13.8 MB of additional disk space will be used.
Get:1 http://deb.debian.org/debian bookworm/main amd64 libpcaudio0 amd64 1.2-2 [8800 B]
Get:2 http://deb.debian.org/debian bookworm/main amd64 libsonic0 amd64 0.2.0-12 [11.0 kB]
Get:3 http://deb.debian.org/debian bookworm/main amd64 espeak-ng-data amd64 1.51+dfsg-10+deb12u2 [4256 kB]
Get:4 http://deb.debian.org/debian bookworm/main amd64 libespeak-ng1 amd64 1.51+dfsg-10+deb12u2 [200 kB]
Get:5 http://deb.debian.org/debian bookworm/main amd64 espeak-ng amd64 1.51+dfsg-10+deb12u2 [353 kB]
Fetched 4829 kB in 0s (52.2 MB/s)
debconf: delaying package configuration, since apt-utils is not installed

78Selecting pr

In [9]:
!pip install .

Processing /__modal/volumes/vo-KmPbOHyRAQmNKh7dlOw3Is/neutts
  Installing build dependencies ... - \ | done
  Getting requirements to build wheel ... - \ done
  Preparing metadata (pyproject.toml) ... - done
  Preparing metadata (setup.py) ... - done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 249.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 261.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 242.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 251.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.7/910.7 kB 253.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 268.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 261.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 248.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [20]:
!python -m examples.basic_example \
  --input_text "My name is Andy. I'm 25 and I just moved to London. The underground is pretty confusing, but it gets me around in no time at all." \
  --ref_audio /mnt/data-vol/13-female-north-young-story-tuyet-trinh-vivi-11s.mp3 \
  --ref_text "Hello, I'm delighted to assist you with our voice services. Choose a voice that resonates with you, and let's begin our creative audio journey together"

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading phonemizer...
Loading backbone from: neuphonic/neutts-nano on cpu ...



In [14]:
%uv pip install fire loguru wandb -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!python examples/finetune.py examples/finetune_config.yaml

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading config from examples/finetune_config.yaml
2026-02-26 05:40:38.480 | INFO     | __main__:main:119 - Logging to: /mnt/prj-vol/neutts/exps/20260226/neutts-finetune-phoaudiobook
Loading checkpoint from neuphonic/neutts-air
`torch_dtype` is deprecated! Use `dtype` instead!
Loading dataset from disk: 100%|██████████████| 56/56 [00:00<00:00, 9078.23it/s]
Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 15164